# 🌍 Global Development Dashboard
**Data source:** [World Bank Open Data API](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392) (no API key needed)
**Stack:** `requests` · `pandas` · `plotly`

This notebook pulls live data from a REST API, cleans and merges it, and builds four interactive visualisations that tell a story about how wealth, health and population have evolved since 1990.

| # | Visual | Question it answers |
|---|--------|---------------------|
| 1 | Animated bubble chart | How are income and life expectancy related, and how did that change over time? |
| 2 | Animated world map | Where has life expectancy improved the most? |
| 3 | Country comparison tool (dropdowns + legend) | How do specific countries compare on any indicator? |
| 4 | Income-group trends | Is the health gap between rich and poor countries closing? |

## 1. Setup

In [1]:
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import copy
from functools import reduce

# --- Black theme: start from Plotly's dark template and force pure-black backgrounds ---
black = copy.deepcopy(pio.templates["plotly_dark"])
black.layout.paper_bgcolor = "#000000"
black.layout.plot_bgcolor = "#000000"
black.layout.geo.bgcolor = "#000000"
black.layout.xaxis.gridcolor = "#2a2a2a"
black.layout.yaxis.gridcolor = "#2a2a2a"
black.layout.font.color = "#e8e8e8"
pio.templates["black"] = black
pio.templates.default = "black"
try:
    import google.colab  # noqa: F401
    pio.renderers.default = "colab"
except ImportError:
    pass  # fall back to Plotly's default renderer outside Colab

START_YEAR, END_YEAR = 1990, 2022

## 2. Extract: fetching data from the API
Two small helper functions handle pagination, errors and timeouts so the rest of the notebook stays clean.

In [2]:
BASE_URL = "https://api.worldbank.org/v2"

def wb_request(path: str, **params) -> list:
    """Call the World Bank API and return all rows across every page."""
    params = {"format": "json", "per_page": 1000, **params}
    rows, page = [], 1
    while True:
        resp = requests.get(f"{BASE_URL}/{path}", params={**params, "page": page}, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        if len(payload) < 2 or payload[1] is None:
            raise ValueError(f"No data returned for {path}: {payload}")
        meta, data = payload
        rows.extend(data)
        if page >= meta["pages"]:
            return rows
        page += 1


def fetch_indicator(code: str, name: str) -> pd.DataFrame:
    """Download one indicator for all countries as a tidy dataframe."""
    rows = wb_request(f"country/all/indicator/{code}", date=f"{START_YEAR}:{END_YEAR}")
    df = pd.DataFrame(
        {
            "iso3": [r["countryiso3code"] for r in rows],
            "year": [int(r["date"]) for r in rows],
            name: [r["value"] for r in rows],
        }
    )
    return df[df["iso3"] != ""]


def fetch_countries() -> pd.DataFrame:
    """Country metadata (region, income group); drops regional aggregates like 'World'."""
    rows = wb_request("country")
    df = pd.DataFrame(
        {
            "iso3": [r["id"] for r in rows],
            "country": [r["name"] for r in rows],
            "region": [r["region"]["value"] for r in rows],
            "income_group": [r["incomeLevel"]["value"] for r in rows],
        }
    )
    return df[df["region"] != "Aggregates"]

## 3. Transform: merging everything into one analysis-ready table

In [3]:
INDICATORS = {
    "SP.DYN.LE00.IN": "life_exp",     # Life expectancy at birth (years)
    "NY.GDP.PCAP.CD": "gdp_per_cap",  # GDP per capita (current US$)
    "SP.POP.TOTL":    "population",   # Total population
}

frames = [fetch_indicator(code, name) for code, name in INDICATORS.items()]
countries = fetch_countries()

df = (
    reduce(lambda a, b: a.merge(b, on=["iso3", "year"], how="outer"), frames)
    .merge(countries, on="iso3", how="inner")   # inner join also removes aggregates
    .sort_values(["country", "year"])
    .reset_index(drop=True)
)

# Pretty labels reused by the charts
LABELS = {
    "life_exp": "Life expectancy (years)",
    "gdp_per_cap": "GDP per capita (US$)",
    "population": "Population",
}
INCOME_ORDER = ["Low income", "Lower middle income", "Upper middle income", "High income"]

print(f"{len(df):,} rows · {df['country'].nunique()} countries · {df['year'].min()}–{df['year'].max()}")
print("\nMissing values (%):")
print((df[list(LABELS)].isna().mean() * 100).round(1).to_string())
df.head()

7,161 rows · 217 countries · 1990–2022

Missing values (%):
life_exp       0.0
gdp_per_cap    4.9
population     0.0


,iso3,year,life_exp,gdp_per_cap,population,country,region,income_group
0,AFG,1990,45.118,NaN,12045660.0,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
1,AFG,1991,45.521,NaN,12238879.0,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
2,AFG,1992,46.569,NaN,13278974.0,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
3,AFG,1993,51.021,NaN,14943172.0,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
4,AFG,1994,50.969,NaN,16250794.0,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income


## 4. Visualisations

### 4.1 Wealth vs. health, 1990 → 2022
Press ▶ or drag the slider. Bubble size = population, colour = region. The log x-axis keeps both poor and rich economies readable.

In [9]:
bubble = df.dropna(subset=["life_exp", "gdp_per_cap", "population"])

fig1 = px.scatter(
    bubble,
    x="gdp_per_cap", y="life_exp",
    size="population", color="region",
    hover_name="country",
    hover_data={"gdp_per_cap": ":,.0f", "life_exp": ":.1f", "population": ":,", "year": False, "region": False},
    animation_frame="year", animation_group="country",
    log_x=True,
    size_max=60,
    range_x=[200, 150_000], range_y=[25, 90],
    labels=LABELS,
    title="Wealth vs. health: life expectancy and GDP per capita",
)
fig1.update_layout(height=600, legend_title_text="Region", legend=dict(orientation="h", y=-0.25))
fig1.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 250
fig1.show()

### 4.2 Life expectancy around the world
Animated choropleth. Hover for exact values.

In [5]:
map_df = df.dropna(subset=["life_exp"])

fig2 = px.choropleth(
    map_df,
    locations="iso3", color="life_exp",
    hover_name="country",
    animation_frame="year",
    color_continuous_scale="Viridis",
    range_color=(40, 85),
    projection="natural earth",
    labels=LABELS,
    title="Life expectancy at birth, by country",
)
fig2.update_geos(
    bgcolor="#000000", showocean=False, showlakes=False,
    showland=True, landcolor="#1a1a1a",
    showcountries=True, countrycolor="#333333",
    showcoastlines=False, showframe=False,
)
fig2.update_layout(height=550, margin=dict(l=0, r=0, t=60, b=0))
fig2.show()

### 4.3 Build-your-own comparison
Use the **dropdown** to switch indicator and the **Linear / Log** buttons to change the scale.
**Click** a country in the legend to show/hide it, **double-click** to isolate it. The 50 most populous countries are included.

In [6]:
DEFAULTS = {"BRA", "USA", "CHN", "IND", "DEU"}
INDICATORS_LIST = list(LABELS)

# 50 most populous countries (latest available year)
top_iso = (
    df.dropna(subset=["population"]).sort_values("year").groupby("iso3").tail(1)
    .nlargest(50, "population")["iso3"].tolist()
)
comp = df[df["iso3"].isin(top_iso)]

palette = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
color_of = {iso: palette[i % len(palette)] for i, iso in enumerate(sorted(top_iso))}

fig3 = go.Figure()
trace_index = []  # (indicator, iso3) for every trace, in order
first_ind = "gdp_per_cap"

for ind in INDICATORS_LIST:
    for iso, g in comp.groupby("iso3"):
        g = g.dropna(subset=[ind])
        if g.empty:
            continue
        show = (ind == first_ind)
        fig3.add_trace(go.Scatter(
            x=g["year"], y=g[ind], mode="lines+markers",
            name=g["country"].iloc[0], legendgroup=iso,
            line=dict(color=color_of[iso], width=2.5), marker=dict(size=5),
            visible=(True if iso in DEFAULTS else "legendonly") if show else False,
            hovertemplate="%{y:,.1f}<extra>%{fullData.name}</extra>",
        ))
        trace_index.append((ind, iso))

def visibility_for(ind):
    return [
        (True if iso in DEFAULTS else "legendonly") if t_ind == ind else False
        for t_ind, iso in trace_index
    ]

fig3.update_layout(
    title=f"{LABELS[first_ind]} over time",
    xaxis_title="Year", yaxis_title=LABELS[first_ind],
    height=600, hovermode="x unified",
    legend=dict(title="Country (click to toggle)"),
    margin=dict(t=110),
    updatemenus=[
        dict(  # indicator selector
            type="dropdown", x=0.0, y=1.15, xanchor="left", yanchor="top",
            bgcolor="#1a1a1a", bordercolor="#555", font=dict(color="#e8e8e8"),
            buttons=[
                dict(label=LABELS[k], method="update",
                     args=[{"visible": visibility_for(k)},
                           {"title.text": f"{LABELS[k]} over time", "yaxis.title.text": LABELS[k]}])
                for k in INDICATORS_LIST
            ],
            active=INDICATORS_LIST.index(first_ind),
        ),
        dict(  # linear / log toggle
            type="buttons", direction="right", x=0.42, y=1.15, xanchor="left", yanchor="top",
            bgcolor="#1a1a1a", bordercolor="#555", font=dict(color="#e8e8e8"),
            buttons=[
                dict(label="Linear", method="relayout", args=[{"yaxis.type": "linear"}]),
                dict(label="Log", method="relayout", args=[{"yaxis.type": "log"}]),
            ],
        ),
    ],
)
fig3.show()

### 4.4 Is the gap closing?
Median life expectancy for each World Bank income group.

In [7]:
gap = (
    df.dropna(subset=["life_exp"])
    .groupby(["income_group", "year"], as_index=False)["life_exp"].median()
)
gap = gap[gap["income_group"].isin(INCOME_ORDER)]

fig4 = px.line(
    gap, x="year", y="life_exp", color="income_group",
    category_orders={"income_group": INCOME_ORDER},
    color_discrete_sequence=["#ef553b", "#fdb42f", "#4cc9f0", "#7cfc9a"],
    labels={"life_exp": "Median life expectancy (years)", "year": "Year", "income_group": "Income group"},
    title="Median life expectancy by income group",
)
fig4.update_traces(line=dict(width=3))
fig4.update_layout(height=500, hovermode="x unified")

# Annotate the gap between richest and poorest groups in the latest year
last = gap[gap["year"] == gap["year"].max()].set_index("income_group")["life_exp"]
diff = last["High income"] - last["Low income"]
fig4.add_annotation(
    x=gap["year"].max(), y=(last["High income"] + last["Low income"]) / 2,
    text=f"Gap: {diff:.1f} years", showarrow=False, xanchor="right",
    font=dict(size=14), bgcolor="rgba(0,0,0,0.7)", bordercolor="#666",
)
fig4.show()

## 5. Key takeaways
Thorough analysis reveal:
- Life expectancy rises steeply with income at low GDP levels, then flattens out (diminishing returns).
- Almost every region moved up and to the right between 1990 and 2022.
- The life-expectancy gap between low- and high-income countries has narrowed (see 4.4), but remains large.

---
*Data: World Bank Open Data (CC BY 4.0). Built with Python, pandas and Plotly.*